# Fase 3 — Semana 2: pipeline de preprocesamiento en clases

**Grupo 4 · MCDI500 · Encuesta Nacional de Salud 2016-2017**

En la Sumativa 1 dejamos listo un conjunto de 5.511 personas para
estudiar cómo se asocian edad, sexo, escolaridad, ingreso y zona con
cinco indicadores de riesgo cardiovascular: hipertensión, diabetes,
colesterol alto, índice de masa corporal y actividad física. Ese
preprocesamiento vivía en funciones sueltas dentro de un notebook.

En esta entrega reescribimos esos mismos pasos como clases que
comparten una interfaz común. El criterio de éxito es concreto: el
pipeline con clases debe entregar exactamente el mismo conjunto que
guardamos en la Fase 2.

| Sección | Contenido |
|---|---|
| 1 | Configuración y carga del conjunto elegible F1-F2 |
| 2 | Pipeline de preprocesamiento en clases |
| 3 | Verificación contra el resultado de la Fase 2 |
| 4 | Validación: caso normal, casos límite y excepciones |
| 5 | Eficiencia: tiempo y memoria |
| 6 | Patrón de diseño Strategy aplicado a la imputación |
| 7 | Arquitectura y conclusiones |

## 1. Configuración y carga del conjunto elegible F1-F2

Partimos del archivo filtrado por ponderador en la Fase 2, antes de
la limpieza. Así las clases tienen que reproducir todo el
preprocesamiento, y el resultado se puede comparar con el conjunto
final guardado en esa fase. Las columnas se agrupan según su rol en
el estudio: predictoras sociodemográficas, indicadores de riesgo y
variables del diseño muestral.

In [ ]:
RUTA_DATOS = "data/processed/ens_variables_f1f2.xlsx"
COLUMNA_ID = "IdEncuesta"

# Predictoras sociodemográficas
COLUMNAS_PREDICTORAS_CONTINUAS = ["Edad", "anos_estudio_MINSAL_1", "as27"]
COLUMNAS_PREDICTORAS_NOMINALES = ["Sexo", "Zona"]
COLUMNAS_PREDICTORAS_ORDINALES = ["as28"]

# Indicadores de riesgo cardiovascular (se analizan por separado)
COLUMNAS_RESULTADO_BINARIAS = ["HTA"]
COLUMNAS_RESULTADO_NOMINALES = ["di3", "dis2"]
COLUMNAS_RESULTADO_ORDINALES = ["GPAQ"]
COLUMNAS_RESULTADO_CONTINUAS = ["IMC"]

# Diseño muestral: se conservan sin transformar
COLUMNAS_DISENO_MUESTRAL = ["Fexp_F1F2p_Corr", "Conglomerado", "Estrato"]

SEMILLA = 2026

COLUMNAS_ESPERADAS = (
    [COLUMNA_ID]
    + COLUMNAS_PREDICTORAS_CONTINUAS + COLUMNAS_PREDICTORAS_NOMINALES
    + COLUMNAS_PREDICTORAS_ORDINALES + COLUMNAS_RESULTADO_BINARIAS
    + COLUMNAS_RESULTADO_NOMINALES + COLUMNAS_RESULTADO_ORDINALES
    + COLUMNAS_RESULTADO_CONTINUAS + COLUMNAS_DISENO_MUESTRAL
)
print("Columnas declaradas:", len(COLUMNAS_ESPERADAS))

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# La raíz se busca aquí porque es la que permite importar src/;
# por eso no puede venir desde el propio src/carga.py.
RAIZ = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".git").exists()), None)
if RAIZ is None:
    raise FileNotFoundError("No se encontró la raíz del repositorio (.git).")
sys.path.append(str(RAIZ / "src"))

from carga import cargar_conjunto, perfilar
from transformador import Transformador

np.random.seed(SEMILLA)
print("pandas", pd.__version__, "· NumPy", np.__version__, "· semilla", SEMILLA)

In [ ]:
datos = cargar_conjunto(RAIZ / RUTA_DATOS, COLUMNAS_ESPERADAS)
perfil = perfilar(datos)
perfil[perfil["nulos"] > 0]

El conjunto tiene 5.520 personas. Los nulos se concentran en `as27`
(995), `GPAQ` (196), `anos_estudio_MINSAL_1` (47), `IMC` (37) y `HTA`
(9). La no respuesta de `as28` no aparece aquí porque está codificada
como -9999: el pipeline debe convertirla en nulo antes de imputar.

## 2. Pipeline de preprocesamiento en clases

Cada paso de limpieza de la Fase 2 se reescribe como una subclase de
`Transformador` (`src/transformador.py`). La clase base fija el orden
de uso: `ajustar()` calcula los parámetros con el conjunto de
referencia y `transformar()` los aplica sobre una copia, sin volver a
calcularlos. Cada subclase solo define qué calcula (`aprender()`) y
cómo lo usa (`aplicar()`). Los pasos concretos de imputación,
codificación y escalamiento se incorporan en las subsecciones
siguientes.